In [2]:
from dotenv import load_dotenv
import os
import time  # <--- agrega esta línea
import json
import requests

# Ruta .env en la raíz del repo
env_path = os.path.abspath(os.path.join(os.getcwd(), "../../..", ".env"))
load_dotenv(dotenv_path=env_path)

# Obtener clave de Grok desde .env
GROK_API_KEY = os.getenv("GROK_API_KEY")

if GROK_API_KEY:
    print("Clave de Grok cargada correctamente.")
else:
    raise ValueError("❌ No se encontró la clave de Grok. Verifica la ruta del .env.")

Clave de Grok cargada correctamente.


In [4]:
def call_grok_api(prompt, text):
    """
    Envía un texto y un prompt al modelo Grok-4 y devuelve el resumen generado.
    """
    url = "https://api.x.ai/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {GROK_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "grok-4",
        "messages": [
            {"role": "system", "content": "Eres un asistente experto en simplificar lenguaje biomédico."},
            {"role": "user", "content": f"{prompt}\n\nTexto:\n{text}"}
        ]
    }

    try:
        start_time = time.time()
        response = requests.post(url, headers=headers, json=payload)
        elapsed = time.time() - start_time

        # Mostrar más información si algo falla
        print(f"Código de estado: {response.status_code}")
        try:
            print("Respuesta (primeros 200 caracteres):", response.text[:200])
        except Exception as e:
            print("No se pudo imprimir el texto:", e)

        if response.status_code == 200:
            data = response.json()
            # Muestra las claves del JSON
            print("Claves en respuesta:", data.keys())
            output = data["choices"][0]["message"]["content"]
            return output.strip(), elapsed
        else:
            print(f"Error HTTP {response.status_code}: {response.text}")
            return None, elapsed
    except Exception as e:
        print("Error en la solicitud:", e)
        return None, None

In [5]:
import pandas as pd
#traemos el archivo test.csv
ruta_dataset = "../data-sources/pre-processed/data_finetuning_test.csv"
df = pd.read_csv(ruta_dataset, encoding="utf-8", on_bad_lines="skip")
display(df.head(2))

,name,article,summary
0,10.1002-14651858.CD009781.pub2,Background\r\nTraumatic corneal abrasions are ...,Topical non‐steroidal anti‐inflammatory drugs ...
1,10.1002-14651858.CD010694.pub2,"Background\r\nVenous leg ulcers are common, ch...",Sulodexide for venous leg ulcers\r\nReview que...


In [6]:
# Prompt a utilizarse
prompt = """Using the following abstract of a biomedical study as input, generate a Plain Language Summary (PLS) understandable by any patient, regardless of their health literacy. Ensure that the generated text adheres to the following instructions which should be followed step-by-step:
    a. Specific Structure: the generated PLS should be presented in a logical order, using the following order:
        1. Plain Title
        2. Rationale
        3. Trial Design
        4. Results
    b. Sections should be authored following these parameters:
        1. Plain Title: Simplified title understandable to a layperson that summarizes the research that was done.
        2. Rationale: Include: background or study rationale providing a general description of the condition, what it may cause or why it is a burden for the patients; the reason and main hypothesis for the study; and why the study is needed, and why the study medication has the potential to treat the condition.
        3. Trial Design: Answer ‘How is this study designed?’ Include the description of the design, description of study and patient population (age, health condition, genre), and the expected amount of time a person will be in the study.
        4. Results: answer ‘What were the main results of the study’, include what are the benefits for the patients, how the study was relevant for the area of study, and what are the conclusions from the investigator.
    c. Consistency and Replicability: the generated PLS should be consistent regardless of the order of sentences or the specific phrasing used in the input protocol text.
    d. Compliance with Plain Language Guidelines: The generated PLS must follow all of these plain language guidelines:
        1. Have readability grade level of 6 or below.
        2. Do not have jargon. All technical or medical words or terms should be defined or broken down into simple and logical explanations.
        3. Active voice, not passive
        4. Mostly one or two syllable words
        5. Sentences of 15 words or less
        6. Short paragraphs of 3-5 sentences
        7. Simple numbers (eg, ratios, no percentages)
    e. Do not invent Content: The AI model should not invent information. If the AI model includes data other than the one given in the input abstract, the AI model should guarantee such data is verified and real.
    f. Aim for an approximate PLS length of 500-900 words.

    Abstract of a biomedical study text:Using the following abstract of a biomedical study as input, generate a Plain Language Summary (PLS) understandable by any patient, regardless of their health literacy. Ensure that the generated text adheres to the following instructions which should be followed step-by-step:
    a. Specific Structure: the generated PLS should be presented in a logical order, using the following order:
        1. Plain Title
        2. Rationale
        3. Trial Design
        4. Results
    b. Sections should be authored following these parameters:
        1. Plain Title: Simplified title understandable to a layperson that summarizes the research that was done.
        2. Rationale: Include: background or study rationale providing a general description of the condition, what it may cause or why it is a burden for the patients; the reason and main hypothesis for the study; and why the study is needed, and why the study medication has the potential to treat the condition.
        3. Trial Design: Answer ‘How is this study designed?’ Include the description of the design, description of study and patient population (age, health condition, genre), and the expected amount of time a person will be in the study.
        4. Results: answer ‘What were the main results of the study’, include what are the benefits for the patients, how the study was relevant for the area of study, and what are the conclusions from the investigator.
    c. Consistency and Replicability: the generated PLS should be consistent regardless of the order of sentences or the specific phrasing used in the input protocol text.
    d. Compliance with Plain Language Guidelines: The generated PLS must follow all of these plain language guidelines:
        1. Have readability grade level of 6 or below.
        2. Do not have jargon. All technical or medical words or terms should be defined or broken down into simple and logical explanations.
        3. Active voice, not passive
        4. Mostly one or two syllable words
        5. Sentences of 15 words or less
        6. Short paragraphs of 3-5 sentences
        7. Simple numbers (eg, ratios, no percentages)
    e. Do not invent Content: The AI model should not invent information. If the AI model includes data other than the one given in the input abstract, the AI model should guarantee such data is verified and real.
    f. Aim for an approximate PLS length of 500-900 words."""

# Tomar las dos primeras filas para prueba inicial
#df_prueba = df.head(2).copy()

# Lista donde guardaremos los resúmenes generados de la prueba
#gen_summaries = []

#for i, fila in df_prueba.iterrows():
#    print(f"\n Procesando {fila['name']} ({i+1}/{len(df_prueba)})...\n")
#    resumen, tiempo = call_grok_api(prompt, fila['article'])
#    gen_summaries.append(resumen if resumen else "")
#    print(f"Tiempo de respuesta: {tiempo:.2f} s\n")

# Añadir columna con los resúmenes generados
#df_prueba["gen_summary"] = gen_summaries

# Guardar en CSV la prueba
#ruta_salida = "./resultados_grok.csv"
#df_prueba.to_csv(ruta_salida, index=False, encoding="utf-8")

#print(f"Resultados guardados en: {os.path.abspath(ruta_salida)}")
#display(df_prueba)

In [5]:
#Aca vamos a usar todo el dataset
df_groktest = df.copy()

# Lista donde guardaremos los resúmenes generados
gen_summaries = []

# Contador de tiempo total
inicio_total = time.time()

for i, fila in df_groktest.iterrows():
    print(f"\n Procesando {fila['name']} ({i+1}/{len(df_groktest)})...\n")
    resumen, tiempo = call_grok_api(prompt, fila['article'])
    gen_summaries.append(resumen if resumen else "")
    print(f" Tiempo de respuesta: {tiempo:.2f} s\n")

# Añadir columna con los resúmenes generados
df_groktest["gen_summary"] = gen_summaries

# Guardar resultados
ruta_salida = "./results_grok2.csv"
df_groktest.to_csv(ruta_salida, index=False, encoding="utf-8")

duracion_total = time.time() - inicio_total
print(f"Resultados guardados en: {os.path.abspath(ruta_salida)}")
print(f"Tiempo total: {duracion_total/60:.2f} minutos ({duracion_total/len(df_groktest):.2f} s por fila en promedio)")


 Procesando 10.1002-14651858.CD009781.pub2 (1/380)...

Código de estado: 200
Respuesta (primeros 200 caracteres): {"id":"738437d6-e6b2-fd8a-ddbe-1f175fcabe2a","object":"chat.completion","created":1762996788,"model":"grok-4-0709","choices":[{"index":0,"message":{"role":"assistant","content":"### Plain Title\nRevie
Claves en respuesta: dict_keys(['id', 'object', 'created', 'model', 'choices', 'usage', 'system_fingerprint'])
 Tiempo de respuesta: 33.91 s


 Procesando 10.1002-14651858.CD010694.pub2 (2/380)...

Código de estado: 200
Respuesta (primeros 200 caracteres): {"id":"8c3749e0-a69b-6128-2ce7-1f84468581b7","object":"chat.completion","created":1762996822,"model":"grok-4-0709","choices":[{"index":0,"message":{"role":"assistant","content":"### Plain Title\nTesti
Claves en respuesta: dict_keys(['id', 'object', 'created', 'model', 'choices', 'usage', 'system_fingerprint'])
 Tiempo de respuesta: 38.80 s


 Procesando 10.1002-14651858.CD009416.pub2 (3/380)...

Código de estado: 200
Respue

In [8]:
# Ruta del archivo csv guardado para verificar su contenido.
ruta_csv = "./results_grok2.csv"
df_check = pd.read_csv(ruta_csv)


df_check.info()

df_check.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   name         380 non-null    object
 1   article      380 non-null    object
 2   summary      380 non-null    object
 3   gen_summary  377 non-null    object
dtypes: object(4)
memory usage: 12.0+ KB


,name,article,summary,gen_summary
0,10.1002-14651858.CD009781.pub2,Background\r\nTraumatic corneal abrasions are ...,Topical non‐steroidal anti‐inflammatory drugs ...,### Plain Title\nReview of Studies on Eye Drop...
1,10.1002-14651858.CD010694.pub2,"Background\r\nVenous leg ulcers are common, ch...",Sulodexide for venous leg ulcers\r\nReview que...,### Plain Title\nTesting if Sulodexide Medicin...
2,10.1002-14651858.CD009416.pub2,Background\r\nThere is currently no strong con...,Which treatments are effective for the treatme...,### Plain Title\nReview of Treatments for Long...
3,10.1002-14651858.CD004104.pub4,Background\r\nNon‐invasive ventilation (NIV) w...,Non‐invasive ventilation for people with respi...,### Plain Title\nTesting a Breathing Machine t...
4,10.1002-14651858.CD012689.pub2,Background\r\nSpace spraying is the dispersal ...,Insecticide space spraying for preventing mala...,### Plain Title\nDoes Spraying Insect Killer i...


In [9]:
# Ver cuántos faltan
faltante = df_check[df_check["gen_summary"].isnull()]
print(f" Faltan {len(faltante)} resúmenes.")
display(faltante[["name", "article"]])

 Faltan 3 resúmenes.


,name,article
115,10.1002-14651858.CD006173.pub2,Background\r\nAdministration of the uterotonic...
214,10.1002-14651858.CD006652.pub5,Background\r\nAnticoagulation may improve surv...
281,10.1002-14651858.CD003408.pub3,Background\r\nSince pulmonary artery balloon f...


In [10]:
# Reprocesar solo esa fila
if not faltante.empty:
    for i, fila in faltante.iterrows():
        print(f"\n Reprocesando {fila['name']}...\n")
        resumen, tiempo = call_grok_api(prompt, fila['article'])
        df_check.loc[i, "gen_summary"] = resumen if resumen else ""
        print(f"Reparado en {tiempo:.2f} s")

# Guardar nuevamente el CSV actualizado
df_check.to_csv(ruta_csv, index=False, encoding="utf-8")
print(f"Archivo actualizado: {os.path.abspath(ruta_csv)}")


 Reprocesando 10.1002-14651858.CD006173.pub2...

Código de estado: 200
Respuesta (primeros 200 caracteres): {"id":"878ac369-7bfa-689d-ad01-5e3c74ca6a00","object":"chat.completion","created":1763079387,"model":"grok-4-0709","choices":[{"index":0,"message":{"role":"assistant","content":"### Plain Title\nDoes 
Claves en respuesta: dict_keys(['id', 'object', 'created', 'model', 'choices', 'usage', 'system_fingerprint'])
Reparado en 49.51 s

 Reprocesando 10.1002-14651858.CD006652.pub5...

Código de estado: 200
Respuesta (primeros 200 caracteres): {"id":"9c5ec478-4777-11ff-28ae-ea7f4eb97df5","object":"chat.completion","created":1763079437,"model":"grok-4-0709","choices":[{"index":0,"message":{"role":"assistant","content":"### Plain Title\nBlood
Claves en respuesta: dict_keys(['id', 'object', 'created', 'model', 'choices', 'usage', 'system_fingerprint'])
Reparado en 37.56 s

 Reprocesando 10.1002-14651858.CD003408.pub3...

Código de estado: 200
Respuesta (primeros 200 caracteres): {"id":"77

In [11]:
# Ruta del archivo csv guardado para verificar su contenido.
ruta_csv = "./results_grok2.csv"
df_check = pd.read_csv(ruta_csv)


df_check.info()

df_check.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   name         380 non-null    object
 1   article      380 non-null    object
 2   summary      380 non-null    object
 3   gen_summary  380 non-null    object
dtypes: object(4)
memory usage: 12.0+ KB


,name,article,summary,gen_summary
0,10.1002-14651858.CD009781.pub2,Background\r\nTraumatic corneal abrasions are ...,Topical non‐steroidal anti‐inflammatory drugs ...,### Plain Title\nReview of Studies on Eye Drop...
1,10.1002-14651858.CD010694.pub2,"Background\r\nVenous leg ulcers are common, ch...",Sulodexide for venous leg ulcers\r\nReview que...,### Plain Title\nTesting if Sulodexide Medicin...
2,10.1002-14651858.CD009416.pub2,Background\r\nThere is currently no strong con...,Which treatments are effective for the treatme...,### Plain Title\nReview of Treatments for Long...
3,10.1002-14651858.CD004104.pub4,Background\r\nNon‐invasive ventilation (NIV) w...,Non‐invasive ventilation for people with respi...,### Plain Title\nTesting a Breathing Machine t...
4,10.1002-14651858.CD012689.pub2,Background\r\nSpace spraying is the dispersal ...,Insecticide space spraying for preventing mala...,### Plain Title\nDoes Spraying Insect Killer i...
